# 02 -- Harmonization compatibility check

Fits a `Harmonizer` over `data.active_datasets` and inspects the resulting common/unique feature split and output width -- run this before Stage A to confirm a config change (adding/removing a dataset) produced the feature layout you expected, and to catch `DatasetIdentityLeakageError` early.


## Setup

Run this cell first. It's the ONLY cell you should need to edit: change
`CONFIG_OVERRIDES` (a list of `--set key.path=value` style dotted overrides,
same syntax as `training.run`'s CLI) to narrow `data.active_datasets`,
switch `architecture`, point at a different Drive folder, etc.


In [ ]:
# ---- Single config cell: this is the only cell you should need to edit ----
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess, os
    REPO_DIR = '/content/dataset_moe_nids'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/selimsidan/dataset_moe_nids', REPO_DIR], check=True)
    %cd $REPO_DIR
    %pip install -q -r requirements.txt

CONFIG_PATH = 'config/default.yaml'

# Dotted --set overrides, same syntax as training.run's CLI. Narrow
# data.active_datasets to 2-3 datasets here for a fast iteration cycle --
# every downstream module (registry, harmonizer, expert-bank sizing,
# checkpoints) adapts automatically, no other code changes needed.
CONFIG_OVERRIDES = [
    # 'data.active_datasets=[NF-UNSW-NB15-v3,NF-BoT-IoT-v3]',
    # 'architecture=moe_dataset_soft',
    # 'training.device=cuda',
]

from training.config import load_config
config = load_config(CONFIG_PATH, CONFIG_OVERRIDES)
print('run_name:', config['run_name'])
print('architecture:', config['architecture'])
print('active_datasets:', config['data']['active_datasets'])
print('checkpoint_dir:', config['training']['checkpoint_dir'])


## Load, split, and fit the harmonizer (mirrors `training.dataset.prepare_datasets`, but stops before tensorizing)

In [ ]:
from training.dataset import prepare_datasets

data = prepare_datasets(config)
h = data.harmonizer
print('output_width:', h.output_width)
print(f'common_features ({len(h.common_features)}):', h.common_features)
print(f'unique_features ({len(h.unique_features)}):', h.unique_features)
print('\nper-dataset presence mask (1 = dataset computes this unique feature):')
for name, mask in h.presence_mask.items():
    print(f'  {name}: {mask.astype(int).tolist()}')


## Sanity-check feature tensor shapes

In [ ]:
print('train features:', data.train.features.shape)
print('val features:  ', data.val.features.shape)
print('test features: ', data.test.features.shape)
print('class_names:', data.class_names)
print('active_datasets:', data.active_datasets)
assert data.train.features.shape[1] == h.output_width
print('\nOK -- harmonized width matches Harmonizer.output_width, ready for Stage A.')
